# 🏋️ Практикум: 60 задач с решениями

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/practicum_60_tasks.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#practice). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Задача 3. Степенной метод

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
def power_iteration(A, iters=200):
    v = np.random.randn(A.shape[0])
    for _ in range(iters):
        v = A @ v; v /= np.linalg.norm(v)
    return v @ A @ v, v           # отношение Рэлея
A = np.array([[4., 1], [2, 3]])
print(power_iteration(A)[0])       # 5.0

### Задача 4. Сжатие изображения через SVD

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
img = np.random.rand(512, 512)                    # подставьте реальное изображение в градациях серого
U, S, Vt = np.linalg.svd(img, full_matrices=False)
approx = lambda k: (U[:, :k] * S[:k]) @ Vt[:k]
energy = np.cumsum(S**2) / np.sum(S**2)           # доля «энергии» для выбора k

### Задача 5. Косинусная близость батчем

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
Q = np.random.randn(100, 384); D = np.random.randn(10_000, 384)
Qn = Q / np.linalg.norm(Q, axis=1, keepdims=True)
Dn = D / np.linalg.norm(D, axis=1, keepdims=True)
S = Qn @ Dn.T                                      # (100, 10000)
top3 = np.argpartition(-S, 3, axis=1)[:, :3]       # O(n) вместо полной сортировки
top3 = np.take_along_axis(top3, np.argsort(-np.take_along_axis(S, top3, 1), 1), 1)

### Задача 6. Проекционная матрица

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
X = np.random.randn(10, 3)
P = X @ np.linalg.solve(X.T @ X, X.T)
print(np.allclose(P @ P, P), np.allclose(P, P.T), round(np.trace(P), 6))   # True True 3.0

### Задача 8. Якобиан softmax

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
z = np.random.randn(4); p = np.exp(z) / np.exp(z).sum()
J = np.diag(p) - np.outer(p, p)
h = 1e-6
J_num = np.array([(np.exp(z + h*e)/np.exp(z + h*e).sum() - np.exp(z - h*e)/np.exp(z - h*e).sum()) / (2*h)
                  for e in np.eye(4)]).T
print(np.allclose(J, J_num, atol=1e-8), np.allclose(J.sum(0), 0))

### Задача 10. Мини-autograd

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import math
class Value:
    def __init__(self, data, parents=(), backward=lambda: None):
        self.data, self.grad, self._parents, self._backward = data, 0.0, parents, backward
    def __add__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        out = Value(self.data + o.data, (self, o))
        def bw(): self.grad += out.grad; o.grad += out.grad
        out._backward = bw; return out
    def __mul__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        out = Value(self.data * o.data, (self, o))
        def bw(): self.grad += o.data * out.grad; o.grad += self.data * out.grad
        out._backward = bw; return out
    def tanh(self):
        t = math.tanh(self.data); out = Value(t, (self,))
        def bw(): self.grad += (1 - t**2) * out.grad
        out._backward = bw; return out
    def backward(self):
        order, seen = [], set()
        def topo(v):
            if v not in seen:
                seen.add(v); [topo(p) for p in v._parents]; order.append(v)
        topo(self); self.grad = 1.0
        for v in reversed(order): v._backward()

x, w, b = Value(0.5), Value(-1.2), Value(0.3)
y = (x * w + b).tanh(); y.backward()
print(y.data, w.grad)   # w.grad = (1 - tanh²) · x

### Задача 14. SGD vs GD экспериментально

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(10_000, 20)); w_true = rng.normal(size=20)
y = (rng.random(10_000) < 1 / (1 + np.exp(-X @ w_true))).astype(float)
sig = lambda z: 1 / (1 + np.exp(-z))
loss = lambda w: np.mean(np.logaddexp(0, X @ w) - y * (X @ w))

def train(bs, lr, epochs=20):
    w = np.zeros(20); hist = []
    for ep in range(epochs):
        idx = rng.permutation(len(y))
        for s in range(0, len(y), bs):
            b = idx[s:s + bs]
            w -= lr * X[b].T @ (sig(X[b] @ w) - y[b]) / len(b)
        hist.append(round(loss(w), 3))
    return hist
print("GD ", train(10_000, 1.0))
print("SGD", train(32, 0.1))

### Задача 15. Gradient clipping

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
def clip_by_global_norm(grads, max_norm):
    total = np.sqrt(sum((g**2).sum() for g in grads))
    scale = min(1.0, max_norm / (total + 1e-6))
    return [g * scale for g in grads], total

### Задача 20. Сэмплирование из softmax через Gumbel

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
z = np.array([1.0, 2.0, 0.5])
u = np.random.rand(200_000, 3)
samples = np.argmax(z - np.log(-np.log(u)), axis=1)
print(np.bincount(samples) / len(samples), np.exp(z) / np.exp(z).sum())   # совпадают

### Задача 23. Репараметризация

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
mu = np.array([1.0, -2.0]); Sigma = np.array([[2.0, 0.8], [0.8, 1.0]])
L = np.linalg.cholesky(Sigma)
x = mu + np.random.randn(100_000, 2) @ L.T
print(x.mean(0).round(2), np.cov(x.T).round(2))

### Задача 33. Логистическая регрессия с нуля

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
X, y = make_classification(2000, 10, random_state=0)
w, b, lam, lr = np.zeros(10), 0.0, 1e-2, 0.5
for _ in range(3000):
    p = 1 / (1 + np.exp(-(X @ w + b)))
    w -= lr * (X.T @ (p - y) / len(y) + lam * w)
    b -= lr * (p - y).mean()
sk = LogisticRegression(C=1 / (lam * len(y))).fit(X, y)   # C = 1/(λN) из-за разной нормировки
print(np.round(w[:4], 3), np.round(sk.coef_[0][:4], 3))   # близки

### Задача 35. Почему residual помогает — численно

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import torch
def grad_norm(residual, depth=50, d=64):
    torch.manual_seed(0)
    layers = [torch.nn.Linear(d, d) for _ in range(depth)]
    x = torch.randn(32, d, requires_grad=True); h = x
    for l in layers:
        h = h + torch.tanh(l(h)) * 0.1 if residual else torch.tanh(l(h))
    h.sum().backward()
    return x.grad.norm().item()
print(grad_norm(False), grad_norm(True))   # без residual — на порядки меньше

### Задача 39. Speculative decoding

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0)
p = np.array([0.5, 0.3, 0.2]); q = np.array([0.2, 0.2, 0.6])
resid = np.maximum(p - q, 0); resid /= resid.sum()
out = []
for _ in range(200_000):
    x = rng.choice(3, p=q)
    out.append(x if rng.random() < min(1, p[x] / q[x]) else rng.choice(3, p=resid))
print(np.bincount(out) / len(out))   # ≈ [0.5, 0.3, 0.2]

### Задача 40. Шаг диффузии

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import torch
T = 1000
betas = torch.linspace(1e-4, 0.02, T); abar = torch.cumprod(1 - betas, 0)

def ddpm_loss(model, x0):
    t = torch.randint(0, T, (x0.shape[0],))
    eps = torch.randn_like(x0)
    a = abar[t].view(-1, *[1] * (x0.dim() - 1))
    xt = a.sqrt() * x0 + (1 - a).sqrt() * eps
    return torch.nn.functional.mse_loss(model(xt, t), eps)

dummy = lambda x, t: torch.zeros_like(x)
print(ddpm_loss(dummy, torch.randn(8, 3, 32, 32)))   # ≈ 1.0: предсказание «нулевого шума» даёт дисперсию ε

### Задача 53. Квантизация INT8

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
W = np.random.randn(256, 512).astype(np.float32)
W[3, 7] = 40.0                                        # выброс в канале 3
def quant(W, per_channel=True):
    m = np.abs(W).max(axis=1, keepdims=True) if per_channel else np.abs(W).max()
    s = m / 127
    return np.clip(np.round(W / s), -127, 127) * s
for pc in [False, True]:
    E = quant(W, pc) - W
    print("поканально" if pc else "на тензор ", np.abs(E).mean().round(4), np.abs(E[3]).mean().round(4))

### Задача 55. Policy gradient на бандите

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0)
mu = np.array([1.0, 1.5, 2.0]) + 10                   # большие награды → большая дисперсия без baseline
def grad_samples(theta, baseline, n=2000):
    p = np.exp(theta - theta.max()); p /= p.sum()
    a = rng.choice(3, n, p=p); r = mu[a] + rng.normal(size=n)
    glogp = np.eye(3)[a] - p                          # ∇θ log softmax
    return glogp * (r - baseline)[:, None]
theta = np.zeros(3)
for b in [0.0, 11.5]:
    g = grad_samples(theta, b)
    print(f"baseline={b:5}: среднее {g.mean(0).round(3)}, дисперсия {g.var(0).sum():.2f}")

### Задача 57. Ранг обновлений при дообучении

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
d, r_true = 512, 8
dW = np.random.randn(d, r_true) @ np.random.randn(r_true, d) + 0.3 * np.random.randn(d, d)
S = np.linalg.svd(dW, compute_uv=False)
energy = np.cumsum(S**2) / np.sum(S**2)
print(np.searchsorted(energy, 0.9) + 1, "из", d)

### Задача 59. Calibration через temperature scaling

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
from scipy.optimize import minimize_scalar
rng = np.random.default_rng(0)
N, K = 5000, 10
y = rng.integers(0, K, N)
logits = rng.normal(size=(N, K)); logits[np.arange(N), y] += 1.5
logits *= 3                                            # искусственная переуверенность
def nll(T):
    z = logits / T; z = z - z.max(1, keepdims=True)
    return -(z[np.arange(N), y] - np.log(np.exp(z).sum(1))).mean()
T = minimize_scalar(nll, bounds=(0.1, 10), method="bounded").x
print(round(T, 2), nll(1.0).round(3), nll(T).round(3))   # T ≈ 2: NLL падает с 1.72 до 1.42